# Model testing

In [13]:
from fire_spread import fire_spread_model, clip_study_area, data_preparation_functions
import geopandas as gpd
import rasterio

In [14]:
# Load full data
fires_gpd = gpd.read_file("../01_Data/06_Wildfire_clusters/Fire_clusters_chaco.shp")

In [15]:
event_id = 20420  # Example event ID
year = fires_gpd[fires_gpd['CLUSTER_ID'] == event_id]['ACQ_DATE'].iloc[0].year

In [16]:
# Get earliest ignition point for the event
ignition_point = fires_gpd[fires_gpd['CLUSTER_ID'] == event_id].sort_values('ACQ_DATE')

In [17]:
# Get ERA5 data for the specific event
era5_data = data_preparation_functions.get_era5_data(
    gdf_init = ignition_point, 
    id_column = 'CLUSTER_ID', 
    lat = None, 
    lon = None, 
    start_date = 'START_TIME', 
    end_date = 'END_TIME', 
    delta_end = 5, 
    delta_start = 5, 
    ee_project = 'webprogrammingumd', 
    event_id = event_id
    )

Taking the first ignition point as representative.
Extracted 264 hourly records for event 20420


In [18]:
path = f"../01_Data/05_Spread_Covariates/ERA5_event_{event_id}.csv"

In [19]:
era5_data[event_id].to_csv(path, index=False)

In [20]:
from fire_spread import monte_carlo as mc
from fire_spread import data_preparation_functions as data_prep
from fire_spread import fire_spread_model as fire_model

Kr = 5
R0 = 0.8
delta_t = 3.5
buffer = 20

In [21]:
# 1. Prepare data
ca_data = data_prep.prepare_ca_inputs(
    event_id=event_id, 
    land_use_path=f'../01_Data/03_MapBiomas/{year}_coverage_lclu_25-1-1.tif',
    weather_csv_path= path,
    srtm_path= '../01_Data/07_SRTM/SRTM_Paraguay_Chaco.tif',
    fire_points_gdf=fires_gpd, buffer_km=buffer)


Preparing CA inputs for event 20420

1. Loading elevation...
  Elevation loaded: (22256, 20369)
  Valid cells: 453332464
  NoData cells: 0
  Elevation range: 0.0 to 623.0 m

2. Finding ignition location...
  Ignition point (full grid): row=6828, col=16263
  Coordinates: x=-58.262300, y=-21.127400
  Ignition time: 2022-09-08 00:00:00

3. Clipping to 20 km buffer around ignition...
  Original grid: 22,256 × 20,369 = 453,332,464 cells (453.3 million)
  Clipped grid:  1,333 × 1,333 = 1,776,889 cells (1.78 million)
  Buffer: 20 km (666 cells)
  Memory reduction: 99.6%
  Ignition point (clipped grid): row=666, col=666

4. Calculating slope and aspect...
  Slope calculated: range 0.0 to 11.3 degrees
  Aspect calculated: range 0 to 6.18 radians

5. Loading land use...
  Land use loaded: (30849, 31147)
  Unique classes: 11

6. Clipping land use to same extent...
  Land use needs snapping before clipping...

7. Mapping to fuel types...
  Fuel type distribution:
    Type 1 (Ks=0.40): 629,502 cel

In [22]:

# 2. Run Monte Carlo (50 realizations)

mc_results = mc.run_monte_carlo_simulation(
    ca_data=ca_data,
    ca_model=fire_model,
    n_runs=50,
    Kr=Kr,
    R0 = R0,
    delta_t = delta_t,
    seed=42  # Reproducibility
)

# 3. Export results
mc.export_monte_carlo_results(
    mc_results,
    output_dir=f'results/event_{event_id}_{Kr}_{R0}_{delta_t}_{buffer}',
    event_id=event_id,
    Kr=Kr
)


Running Monte Carlo Simulation: 50 realizations
Parameters: Kr=5, max_time_steps=1000

Running simulations...


Progress:   2%|▏         | 1/50 [04:14<3:27:53, 254.56s/it]

Fire extinguished at time step 707
  Run 1: 38980 cells burned


Progress:   4%|▍         | 2/50 [10:45<4:27:41, 334.60s/it]

Fire extinguished at time step 699


...

Fire extinguished at time step 151


Progress:  82%|████████▏ | 41/50 [6:16:21<48:21, 322.35s/it]  

Fire extinguished at time step 736
Fire extinguished at time step 1
  Run 40: 1 cells burned
Fire extinguished at time step 12


Progress:  82%|████████▏ | 41/50 [6:28:48<1:25:20, 568.99s/it]


KeyboardInterrupt: 